# Packages

In [1]:
import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime, timedelta, date
import math

import statsmodels.formula.api as smf
import statsmodels.api as sm
from statsmodels.graphics.tsaplots import plot_pacf

import scipy.stats as stats
from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.decomposition import PCA
from sklearn.preprocessing import RobustScaler, LabelEncoder, OrdinalEncoder
import gc
pd.set_option('display.max_columns', 500)
pd.set_option('display.max_rows', 500)
pd.set_option('display.float_format', lambda x: "%.4f" % x)

plt.style.use('ggplot')
sns.set_style('darkgrid')

In [2]:
# helper functions
def get_info(df):
    missing_values_train = pd.DataFrame({'Feature': df.columns,
                              'No. of Missing Values': df.isnull().sum().values,
                              '% of Missing Values': ((df.isnull().sum().values)/len(df)*100)})

    unique_values = pd.DataFrame({'Feature': df.columns,
                                'No. of Unique Values': df.nunique().values})

    feature_types = pd.DataFrame({'Feature': df.columns,
                                'DataType': df.dtypes})

    merged_df = pd.merge(missing_values_train, unique_values, on='Feature', how='left')
    merged_df = pd.merge(merged_df, feature_types, on='Feature', how='left')

    return merged_df

def reduce_memory_usage(df: pd.DataFrame, verbose=False) -> pd.DataFrame:
    start_mem = df.memory_usage(deep=True).sum() / 1024**2
    if verbose:
        print(f"Memory usage before: {start_mem:.2f} MB")
    
    for col in df.columns:
        col_type = df[col].dtype
        
        if col_type == 'float64':
            df[col] = pd.to_numeric(df[col], downcast='float')
        elif col_type == 'int64':
            df[col] = pd.to_numeric(df[col], downcast='integer')
    
    end_mem = df.memory_usage(deep=True).sum() / 1024**2
    if verbose:
        print(f"Memory usage after: {end_mem:.2f} MB")
        print(f"Reduced by {(start_mem - end_mem) / start_mem * 100:.1f}%")
    
    return df

# Pulling Data

In [3]:
dir_path = "./data/NIBRS/output/"
arrestee_path = dir_path + "arrestee_segment_2024.parquet"
groupb_arrest_path = dir_path + "groupb_arrest_segment_2024.parquet"

# admin_path = dir_path + "administrative_segment_2024.parquet"
# offense_path = dir_path + "offense_segment_2024.parquet"
# victim_path = dir_path + "victim_segment_2024.parquet"

arrestee_df = pd.read_parquet(arrestee_path)
arrestee_df = reduce_memory_usage(arrestee_df, verbose=True)
groupb_arrest_df = pd.read_parquet(groupb_arrest_path)
groupb_arrest_df = reduce_memory_usage(groupb_arrest_df, verbose=True)


Memory usage before: 775.68 MB
Memory usage after: 615.24 MB
Reduced by 20.7%
Memory usage before: 561.50 MB
Memory usage after: 430.17 MB
Reduced by 23.4%


In [4]:
print(arrestee_df.shape)
arrestee_df.head()

(3579507, 22)


,segment_level,state_code,ori,incident_number,incident_date,arrestee_sequence_number,arrest_transaction_number,arrest_date,type_of_arrest,multiple_arrestee_segment_indicator,ucr_arrest_offense_code,type_weapon_involved1,automatic_weapon_indicator1,type_weapon_involved2,automatic_weapon_indicator2,age_of_arrestee,sex_of_arrestee,race_of_arrestee,ethnicity_of_arrestee,residence_status_of_arrestee,disposition_arrestee_under_18,db_id
0,6,50,AK0010200,CE0BAA3C728N,20240815,1,55086,20241228,T,N,26A,1,NaN,NaN,NaN,26,M,I,N,R,NaN,2024_1
1,6,50,AK0010200,CE0BAAB-728N,20240815,1,54511,20240815,O,N,13B,1,NaN,NaN,NaN,40,M,I,N,N,NaN,2024_2
2,6,50,AK0010200,CE0BAAML728N,20240813,1,56308,20250220,T,N,520,1,NaN,NaN,NaN,30,M,I,N,N,NaN,2024_3
3,6,50,AK0010200,CE0BAAMM728N,20240813,1,54484,20240813,O,N,13A,1,NaN,NaN,NaN,45,M,W,N,R,NaN,2024_4
4,6,50,AK0010200,CE0BAAQC728N,20240809,1,56142,20250112,T,M,26A,1,NaN,NaN,NaN,26,M,A,N,R,NaN,2024_5


In [5]:
print(groupb_arrest_df.shape)
groupb_arrest_df.head()

(2929868, 20)


,segment_level,state_code,ori,incident_number,arrest_date,arrestee_sequence_number,city_submission,type_of_arrest,ucr_arrest_offense_code,type_weapon_involved1,automatic_weapon_indicator1,type_weapon_involved2,automatic_weapon_indicator2,age_of_arrestee,sex_of_arrestee,race_of_arrestee,ethnicity_of_arrestee,residence_status_of_arrestee,disposition_arrestee_under_18,db_id
0,7,50,AK0010200,0F920BRVSCTD,20240412,1,NaN,T,90Z,1,NaN,NaN,NaN,59,M,W,N,N,NaN,2024_1
1,7,50,AK0010200,0F9G0BRVSCTD,20240430,1,NaN,O,90C,1,NaN,NaN,NaN,30,M,B,N,R,NaN,2024_2
2,7,50,AK0010200,0F9N0BRVSCTD,20240427,1,NaN,O,90J,1,NaN,NaN,NaN,23,M,I,N,R,NaN,2024_3
3,7,50,AK0010200,0F9W0BRVSCTD,20240407,1,NaN,O,90D,1,NaN,NaN,NaN,25,F,W,N,R,NaN,2024_4
4,7,50,AK0010200,0F9X0BRVSCTD,20240423,1,NaN,O,90D,1,NaN,NaN,NaN,37,M,W,N,N,NaN,2024_5


In [6]:
arrestee_df = arrestee_df.assign(is_groupb = 0)
groupb_arrest_df = groupb_arrest_df.assign(is_groupb = 1)

data_df = pd.concat([arrestee_df, groupb_arrest_df])
data_df.head().T

,0,1,2,3,4
segment_level,6,6,6,6,6
state_code,50,50,50,50,50
ori,AK0010200,AK0010200,AK0010200,AK0010200,AK0010200
incident_number,CE0BAA3C728N,CE0BAAB-728N,CE0BAAML728N,CE0BAAMM728N,CE0BAAQC728N
incident_date,20240815.0000,20240815.0000,20240813.0000,20240813.0000,20240809.0000
arrestee_sequence_number,1,1,1,1,1
arrest_transaction_number,55086,54511,56308,54484,56142
arrest_date,20241228,20240815,20250220,20240813,20250112
type_of_arrest,T,O,T,O,T
multiple_arrestee_segment_indicator,N,N,N,N,M


In [8]:
data_df[lambda x: x.ori == "NJ0111100"]

,segment_level,state_code,ori,incident_number,incident_date,arrestee_sequence_number,arrest_transaction_number,arrest_date,type_of_arrest,multiple_arrestee_segment_indicator,ucr_arrest_offense_code,type_weapon_involved1,automatic_weapon_indicator1,type_weapon_involved2,automatic_weapon_indicator2,age_of_arrestee,sex_of_arrestee,race_of_arrestee,ethnicity_of_arrestee,residence_status_of_arrestee,disposition_arrestee_under_18,db_id,is_groupb,city_submission


## Cleaning Citations and Summons

In [ ]:
print(f"Before Removing Citations: {data_df.shape}")
data_df = data_df[lambda x: x.type_of_arrest != 'S']
print(f"After Removing Citations: {data_df.shape}")

data_df["type_of_arrest"].value_counts()

In [ ]:
data_df["incident_date"] = pd.to_datetime(data_df["incident_date"], format="%Y%m%d")
data_df["arrest_date"] = pd.to_datetime(data_df["arrest_date"], format="%Y%m%d")
data_df["incident_date"] = data_df["incident_date"].fillna(data_df["arrest_date"])

In [ ]:
data_dir_path = "./data/nibrs.csv"
data_df.to_csv(data_dir_path, index=False)

# EDA

## Inital Checks

In [ ]:
info_df = get_info(data_df)
info_df

## Distribution of Race and Ethnicity

In [ ]:
data_df["race_of_arrestee"].value_counts()

In [ ]:
data_df["race_of_arrestee"].hist()

In [ ]:
data_df[lambda x: (x.ethnicity_of_arrestee.isna()) & (x.race_of_arrestee == 'U')]

In [ ]:
data_df["ethnicity_of_arrestee"] = data_df["ethnicity_of_arrestee"].fillna("U")
data_df["ethnicity_of_arrestee"].value_counts()

In [ ]:
data_df["ethnicity_of_arrestee"].hist()

In [ ]:
data_df = data_df.assign(
    race_ethnicity_arrestee = lambda x: 
            np.where(
                x.ethnicity_of_arrestee == 'H',
                x.ethnicity_of_arrestee,
                x.race_of_arrestee
            )
)

In [ ]:
any(data_df["race_ethnicity_arrestee"].isna())

In [ ]:
data_df["race_ethnicity_arrestee"].hist()

In [ ]:
data_df["race_of_arrestee"].value_counts()

In [ ]:
data_df["race_ethnicity_arrestee"].value_counts()

In [ ]:
data_df[lambda x: x.race_ethnicity_arrestee == "U"]["ethnicity_of_arrestee"].value_counts()

In [ ]:
(
    data_df
        [lambda x: (x.ethnicity_of_arrestee == 'U') & (x.race_of_arrestee == 'W')]
        .shape
)

In [ ]:
(
    data_df
        [lambda x: (x.ethnicity_of_arrestee == 'U') & (x.race_of_arrestee == 'U')]
        .shape
)

### Distribution by UCR Arrest Offense Code

In [ ]:
(
    data_df
        ["ucr_arrest_offense_code"]
        .value_counts()
        .sort_values(ascending=False)
        .plot(kind="bar")
)

In [ ]:
ordered_indices = (
    data_df["ucr_arrest_offense_code"]
    .value_counts()
    .sort_values(ascending=False)
    .index.tolist()
)

fig, ax = plt.subplots(figsize=(14, 6))
for race in sorted(data_df["race_ethnicity_arrestee"].unique()):
    subset = data_df[data_df["race_ethnicity_arrestee"] == race]
    counts = subset["ucr_arrest_offense_code"].value_counts(normalize=True).reindex(ordered_indices, fill_value=0)
    ax.plot(ordered_indices, counts.values, label=race, alpha=0.7)
ax.set_xticks(range(len(ordered_indices)))
ax.set_xticklabels(ordered_indices, rotation=90)
ax.set_ylabel("Density")
ax.legend()
plt.tight_layout()
plt.show()

**Observations:**
- U:
    - Density of 90D (drunken driving) is higher than all other races
    - Perhaps this makes sense from first principles

In [ ]:
(
    data_df
        [lambda x: (x.race_ethnicity_arrestee != 'U')]
        ["ucr_arrest_offense_code"]
        .value_counts()
        .plot(kind="bar")
)

In [ ]:
(
    data_df
        [lambda x: (x.race_ethnicity_arrestee == 'U')]
        ["ucr_arrest_offense_code"]
        .value_counts()
        .plot(kind="bar")
)

In [ ]:
# Filter only 90D offenses
df_90d = data_df[data_df["ucr_arrest_offense_code"] == "90D"]

# Count by race for 90D, normalize to relative density
race_counts = (
    df_90d["race_ethnicity_arrestee"]
    .value_counts(normalize=True)
    .sort_index()
    .reset_index()
)
race_counts.columns = ["race_ethnicity", "relative_density"]

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(race_counts["race_ethnicity"], race_counts["relative_density"], edgecolor="black")
ax.set_xlabel("Race/Ethnicity")
ax.set_ylabel("Relative Density")
ax.set_title("Relative Density of 90D (Drunken Driving) by Race/Ethnicity")
plt.tight_layout()
plt.show()

### Distribution by States

In [ ]:
state_code_dict = {
    1: "Alabama",
    2: "Arizona",
    3: "Arkansas",
    4: "California",
    5: "Colorado",
    6: "Connecticut",
    7: "Delaware",
    8: "District of Columbia",
    9: "Florida",
    10: "Georgia",
    11: "Idaho",
    12: "Illinois",
    13: "Indiana",
    14: "Iowa",
    15: "Kansas",
    16: "Kentucky",
    17: "Louisiana",
    18: "Maine",
    19: "Maryland",
    20: "Massachusetts",
    21: "Michigan",
    22: "Minnesota",
    23: "Mississippi",
    24: "Missouri",
    25: "Montana",
    26: "Nebraska",
    27: "Nevada",
    28: "New Hampshire",
    29: "New Jersey",
    30: "New Mexico",
    31: "New York",
    32: "North Carolina",
    33: "North Dakota",
    34: "Ohio",
    35: "Oklahoma",
    36: "Oregon",
    37: "Pennsylvania",
    38: "Rhode Island",
    39: "South Carolina",
    40: "South Dakota",
    41: "Tennessee",
    42: "Texas",
    43: "Utah",
    44: "Vermont",
    45: "Virginia",
    46: "Washington",
    47: "West Virginia",
    48: "Wisconsin",
    49: "Wyoming",
    50: "Alaska",
    51: "Hawaii",
    52: "Canal Zone",
    53: "Puerto Rico",
    54: "American Samoa",
    55: "Guam",
    62: "Virgin Islands"
}

In [ ]:
data_df = data_df.assign(state_name = lambda x: x.state_code.map(state_code_dict))
data_df

In [ ]:
(
    data_df
        ["state_name"]
        .value_counts()
        .sort_values(ascending=False)
        .plot(kind="bar")
)

In [ ]:
(
    data_df
        [lambda x: (x.race_ethnicity_arrestee == 'U')]
        ["state_name"]
        .value_counts()
        .sort_values(ascending=False)
        .plot(kind="bar")
)

#### Why Minnesota


In [ ]:
(
    data_df
        [lambda x: (x.race_ethnicity_arrestee != 'U') & (x.state_name == "Minnesota")]
        ["incident_date"]
        .hist()
)

In [ ]:
(
    data_df
        [lambda x: (x.race_ethnicity_arrestee == 'U') & (x.state_name == "Minnesota")]
        ["incident_date"]
        .hist()
)

In [ ]:
(
    data_df
        [lambda x: (x.race_ethnicity_arrestee == 'U') & (x.state_name == "Minnesota")]
        ["ucr_arrest_offense_code"]
        .value_counts()
        .plot(kind="bar")
)

In [ ]:
(
    data_df
        [lambda x: (x.race_ethnicity_arrestee != 'U') & (x.state_name == "Minnesota")]
        ["ucr_arrest_offense_code"]
        .value_counts()
        .plot(kind="bar")
)

## Temporal Data

TODO: 
- Look at the consecutive gaps between reports per agency; we care more about deviation from E[len(gap)]
- Conditioned on a report getting filed, then no missing reports for that day
- Can also look at deviation from E[# of reports on a given day]

In [ ]:
data_df["ori"].nunique()

In [ ]:
data_df.head().T

In [ ]:
data_df = data_df.assign(
    month= lambda x: x.incident_date.dt.month,
    day= lambda x: x.incident_date.dt.day,
    day_of_week= lambda x: x.incident_date.dt.dayofweek
)

In [ ]:
sns.histplot(data=data_df, x="incident_date")
plt.show()

In [ ]:
(
    data_df
        [lambda x: ~x.incident_date.between(datetime(2024,1,1), datetime(2024,12,31), inclusive="both")]
        .sort_values(by="incident_date")
)

TODO: look into these examples above

In [ ]:
data_df_filtered = (
    data_df
        [lambda x: x.incident_date.between(datetime(2024,1,1), datetime(2024,12,31), inclusive="both")]
        .sort_values(by="incident_date")
)

In [ ]:
data_df_filtered

In [ ]:
data_df_filtered["incident_date"].hist()

### Example ORI: AK0010200

In [ ]:
df = data_df_filtered[lambda x: x.ori == 'AK0010200']
df

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(df["incident_date"], bins=30, edgecolor="black", linewidth=0.8)
ax.set_xlabel("Incident Date")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3))
sns.stripplot(data=df, x="incident_date", ax=ax, size=4, linewidth=0.5, jitter=True)
ax.set_xlabel("Incident Date")
plt.tight_layout()
plt.show()

### Example ORI: IL0169400

In [ ]:
df = data_df_filtered[lambda x: x.ori == 'IL0169400']
df.head()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(df["incident_date"], bins=30, edgecolor="black", linewidth=0.8)
ax.set_xlabel("Incident Date")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3))
sns.stripplot(data=df, x="incident_date", ax=ax, size=4, linewidth=0.5, jitter=True)
ax.set_xlabel("Incident Date")
plt.tight_layout()
plt.show()

### Start of Year

In [ ]:
num_unique_incident_dates_df = (
    data_df_filtered
        .groupby("ori")
        ["incident_date"]
        .nunique()
        .reset_index()
        .rename(columns={"incident_date" : "incident_date_count"})
        .sort_values(by="incident_date_count")
)
num_unique_incident_dates_df

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(num_unique_incident_dates_df["incident_date_count"], bins=30, edgecolor="black", linewidth=0.8)
ax.set_xlabel("Incident Date Count")
ax.set_ylabel("Frequency")
plt.tight_layout()
plt.show()

In [ ]:
num_unique_incident_dates_df.describe()

In [ ]:
(
    data_df_filtered
        .assign(is_jan=lambda x: x.month == 1)
        .groupby("ori")["is_jan"]
        .sum()
        .reset_index(name="row_count")
        .sort_values("row_count")
        .set_index("ori")
        # .tail(20)  # just plot the 20 highest, for legibility
        # .plot(kind='bar')
)

In [ ]:
(
    data_df_filtered
        .assign(is_jan=lambda x: x.month == 1)
        .groupby("ori")
        .agg(
            row_count=("is_jan", "sum"),
            unique_dates=("incident_date", "nunique")
        )
        .query("unique_dates >= 100")
        .sort_values("row_count")
        .drop(columns="unique_dates")
)


In [ ]:
zero_jan_reports = (
    data_df_filtered
        .assign(is_jan=lambda x: x.month == 1)
        .groupby("ori")
        .agg(
            row_count=("is_jan", "sum"),
            unique_dates=("incident_date", "nunique")
        )
        .query("unique_dates >= 100")
        .sort_values("row_count")
        .drop(columns="unique_dates")
        .query("row_count == 0")
        .reset_index()
)


In [ ]:
zero_jan_reports

### Example ORI: IN0890100

In [ ]:
df = data_df_filtered[lambda x: x.ori == 'IN0890100']
df

In [ ]:
df.shape

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(df["incident_date"], bins=30, edgecolor="black", linewidth=0.8)
ax.set_xlabel("Incident Date")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

### Example ORI: CA0390200
This seems so wrong
https://www.lodi.gov/392/Statistics

In [ ]:
df = data_df_filtered[lambda x: x.ori == 'CA0390200']
df

In [ ]:
df.shape

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(df["incident_date"], bins=30, edgecolor="black", linewidth=0.8)
ax.set_xlabel("Incident Date")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

### Graphing for all 34 ORIs

In [ ]:
to_graph_oris = zero_jan_reports["ori"].to_list()

In [ ]:
from matplotlib.backends.backend_pdf import PdfPages

output_path = "all_jan_incident_histograms.pdf"

with PdfPages(output_path) as pdf:
    for ori in to_graph_oris:
        df = data_df_filtered.loc[lambda x: x.ori == ori]
        fig, ax = plt.subplots(figsize=(11, 6))
        ax.hist(df["incident_date"], bins=30, edgecolor="black", linewidth=0.8)
        ax.set_title(f"Incident Date Distribution — ORI: {ori} - Len: {len(df)}")
        ax.set_xlabel("Incident Date")
        ax.set_ylabel("Count")
        plt.tight_layout()
        pdf.savefig(fig)
        plt.close(fig)

print(f"Saved to {output_path}")

### Example ORI: CA0194200
Los Angeles Police Department

In [ ]:
df = data_df_filtered[lambda x: x.ori == 'CA0194200']
df

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(df["incident_date"], bins=30, edgecolor="black", linewidth=0.8)
ax.set_xlabel("Incident Date")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

### Empty for Each Month

In [ ]:
zero_month_reports_list = []

for i in range(1, 13):
    temp = (
        data_df_filtered
            .assign(is_specific_month=lambda x: x.month == i)
            .groupby("ori")
            .agg(
                row_count=("is_specific_month", "sum"),
                unique_dates=("incident_date", "nunique")
            )
            .query("unique_dates >= 100")
            .sort_values("row_count")
            .drop(columns="unique_dates")
            .query("row_count == 0")
            .reset_index()
    )
    zero_month_reports_list.append(temp)

zero_month_reports = pd.concat(zero_month_reports_list).drop_duplicates()
for t in zero_jan_reports:
    del t
gc.collect()


In [ ]:
zero_month_reports.reset_index(drop=True, inplace=True)
zero_month_reports.shape

In [ ]:
to_graph_oris = zero_month_reports["ori"].to_list()

In [ ]:
from matplotlib.backends.backend_pdf import PdfPages

output_path = "all_incident_histograms.pdf"

with PdfPages(output_path) as pdf:
    for ori in to_graph_oris:
        df = data_df_filtered.loc[lambda x: x.ori == ori]
        fig, ax = plt.subplots(figsize=(11, 6))
        ax.hist(df["incident_date"], bins=30, edgecolor="black", linewidth=0.8)
        ax.set_title(f"Incident Date Distribution — ORI: {ori} - Len: {len(df)}")
        ax.set_xlabel("Incident Date")
        ax.set_ylabel("Count")
        plt.tight_layout()
        pdf.savefig(fig)
        plt.close(fig)

print(f"Saved to {output_path}")

In [ ]:
zero_month_reports["ori"]

## Incident Number

In [ ]:
data_df_filtered["incident_number"].value_counts().reset_index()["count"].sort_values().value_counts()

In [ ]:
data_df_filtered["incident_number"].value_counts().head(10)

### CE-1XZ8U728N

In [ ]:
data_df_filtered[lambda x: x.incident_number == 'CE-1XZ8U728N'].head()

In [ ]:
data_df_filtered[lambda x: x.incident_number == 'CE-1XZ8U728N']["ori"].nunique()

### Num Unique (ORI, Incident #) pairs

In [ ]:
data_df_filtered[["ori", "incident_number"]].value_counts().head(10)

### CE0ASCPU728N

https://en.wikipedia.org/wiki/2024_University_of_California,_Los_Angeles_pro-Palestinian_campus_occupation

In [ ]:
data_df_filtered[lambda x: (x.ori == "CA0199700") & (x.incident_number == "CE0ASCPU728N")]

In [ ]:
data_df_filtered[lambda x: (x.ori == "CA0199700") & (x.incident_number == "CE0ASCPU728N") 
                            & (x.race_of_arrestee == 'U') & (x.ethnicity_of_arrestee == 'U')]

In [ ]:
data_df_filtered[lambda x: (x.ori == "CA0199700") & (x.incident_number == "CE0ASCPU728N") 
                            & (x.race_of_arrestee == 'U') & (x.ethnicity_of_arrestee == 'N')]